In [50]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
import torch

In [87]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from torch.utils.data import WeightedRandomSampler
from sklearn.metrics import confusion_matrix

# load data
X, y = [], []
for file in os.listdir("../preprocessed_data"):
    if file.endswith(".npy"):
        label = int(file.split("_")[3])  # from e.g. ..._label_0_stripped.npy
        vol = np.load(os.path.join("../preprocessed_data", file))  # shape: (1, D, H, W)
        X.append(vol)
        y.append(label)

# horizontal flip along width axis (W = axis 3)
X_aug = []
y_aug = []
for i in range(len(X)):
    flipped = np.flip(X[i], axis=3)  # width flip
    X_aug.append(flipped)
    y_aug.append(y[i])

X = np.concatenate([X, np.array(X_aug)])
y = np.concatenate([y, np.array(y_aug)])

# flip vertically (height axis)
X_flip_v = [np.flip(x, axis=2) for x in X]
y_flip_v = y.copy()

# add Gaussian noise
X_noise = [x + np.random.normal(0, 0.01, x.shape) for x in X]
y_noise = y.copy()

# merge all
X = np.concatenate([X, np.array(X_flip_v), np.array(X_noise)])
y = np.concatenate([y, np.array(y_flip_v), np.array(y_noise)])

X = np.array(X)
y = np.array(y)

# normalize
X = (X - np.mean(X)) / np.std(X)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

# sample weights
class_sample_count = np.array([np.sum(y_train == t) for t in np.unique(y_train)])
weights = 1. / class_sample_count
sample_weights = weights[y_train]  # assign weight to each sample
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


# dataloader
train_loader = DataLoader(
    TensorDataset(X_train_tensor, y_train_tensor),
    batch_size=2,
    sampler=sampler  # no shuffle needed
)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=2)

# 3D CNN (small)
class Improved3DCNN(nn.Module):
    def __init__(self):
        super(Improved3DCNN, self).__init__()
        self.conv1 = nn.Conv3d(1, 4, 3, padding=1)
        self.bn1 = nn.BatchNorm3d(4)
        self.conv2 = nn.Conv3d(4, 8, 3, padding=1)
        self.bn2 = nn.BatchNorm3d(8)
        self.conv3 = nn.Conv3d(8, 16, 3, padding=1)
        self.bn3 = nn.BatchNorm3d(16)
        self.pool = nn.MaxPool3d(2)
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(16 * 8 * 16 * 16, 32)
        self.fc2 = nn.Linear(32, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.pool(torch.relu(self.bn3(self.conv3(x))))
        x = x.view(x.size(0), -1)
        x = self.dropout(torch.relu(self.fc1(x)))
        return self.fc2(x)

# setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Improved3DCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# train
for epoch in range(10):
    model.train()
    total_loss = 0.0
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1} | Loss: {total_loss / len(train_loader):.4f}")

# test

model.eval()
correct = 0
total = 0
y_true = []
y_pred = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs = inputs.to(device)
        outputs = model(inputs)
        _, predicted = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(predicted.cpu().numpy())

# accuracy & confusion matrix
y_true = np.array(y_true)
y_pred = np.array(y_pred)
accuracy = np.mean(y_pred == y_true)

print(f"Test Accuracy: {accuracy:.2%}")
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Epoch 1 | Loss: 0.7449
Epoch 2 | Loss: 0.6624
Epoch 3 | Loss: 0.5488
Epoch 4 | Loss: 0.5003
Epoch 5 | Loss: 0.5363
Epoch 6 | Loss: 0.4142
Epoch 7 | Loss: 0.3454
Epoch 8 | Loss: 0.3043
Epoch 9 | Loss: 0.2497
Epoch 10 | Loss: 0.2417
Test Accuracy: 86.27%
Confusion matrix:
 [[13  4]
 [ 3 31]]


In [ ]:
df = pd.read_csv("../labelled_patients.csv")

labels_dict = dict(zip(df['SubjectID'], df['Class']))

{46: 1,
 41: 1,
 77: 1,
 48: 1,
 70: 1,
 5: 1,
 12: 1,
 71: 1,
 76: 1,
 49: 1,
 40: 1,
 47: 1,
 78: 1,
 4: 1,
 13: 1,
 25: 1,
 38: 1,
 36: 1,
 31: 1,
 65: 1,
 62: 1,
 30: 1,
 37: 1,
 39: 1,
 52: 1,
 63: 1,
 64: 1,
 8: 1,
 27: 1,
 6: 1,
 11: 1,
 29: 1,
 42: 1,
 45: 1,
 73: 1,
 74: 1,
 28: 1,
 7: 1,
 10: 1,
 26: 1,
 9: 1,
 75: 1,
 72: 1,
 44: 1,
 43: 1,
 66: 1,
 50: 1,
 68: 1,
 32: 1,
 35: 1,
 69: 1,
 51: 1,
 67: 1,
 34: 1,
 33: 1,
 24: 0,
 23: 0,
 15: 0,
 2: 0,
 14: 0,
 3: 0,
 22: 0,
 54: 0,
 53: 0,
 55: 0,
 20: 0,
 18: 0,
 16: 0,
 1: 0,
 17: 0,
 19: 0,
 21: 0,
 61: 0,
 59: 0,
 57: 0,
 56: 0,
 58: 0,
 60: 0}

In [ ]:
import numpy as np

data = np.load("../preprocessed_data/volume_03_label_0_stripped.npy", allow_pickle=True)
for i in range(1, data.shape[0]):
#     plt.imshow(data[i])
#     plt.show()
    plt.imshow(data[0][:, i, :])
    plt.show()

[]


NameError: name 'files' is not defined